# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vishal-141206/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding chosen #1 — "What Predicts Health?" (ML Appendix, Feature Importance, p.26).**
The paper trains a Random Forest to predict a composite "health score" from features including
Average Position, Impressions, Scroll Depth, and CTR — and reports Average Position as the top
driver (importance 43). To its credit, the paper is transparent about a real limitation:
*"the target itself is partly constructed from some of these inputs, so importance is
descriptive rather than causal."* Health Score is defined elsewhere as `Impressions (30pts) +
Position (30pts) + CTR (20pts) + Scroll Depth (20pts)` — meaning two of the "predictive"
features (Impressions, Position) are literally summed into the target itself.

**Methodology question (constructive, not a takedown):** given that the target is a weighted
sum of some of the input features, does "feature importance" here tell us anything beyond
"the model rediscovered the target's own formula"? A useful next check would be re-running
importance after excluding the features that directly compose the target (Impressions,
Position, CTR, Scroll Depth) and seeing what — if anything — the remaining features
(Content Age, Word Count, Sessions) contribute. The paper's own honesty about this ("descriptive
rather than causal") is exactly right, and this suggests a natural follow-up rather than a flaw
in the writing.

**Finding chosen #2 — ML Pipeline methodology (Methodology & Limitations, p.36).**
The ML appendix (K-Means, Random Forest, Logistic Regression, Decision Tree) uses an "80/20
split" for each model, with no mention of grouping by client/brand. The dataset spans 57
brands; if the 80/20 split is a random row split rather than grouped by brand, the same brand's
content could appear in both train and test.

**Methodology question:** does the 80/20 split control for brand grouping? If not, a model can
partially learn brand-specific baseline behavior (e.g., "content from Brand X tends to score
around Y") rather than a generalizable content-level pattern — inflating validation numbers in
a way that wouldn't hold on a genuinely new brand. This is worth checking explicitly, precisely
because it's an easy, understandable mistake — this is the same check I apply to my own model
below in §2, where I found it made a real, measurable difference.

Both questions are asked in the spirit the paper itself models: it already demotes its own
weaker constructs (e.g. explicitly limiting the 361+ freshness bucket's 283:1 ratio due to n=1
decliner) and states its evidence standard plainly. These are the same category of question,
aimed at strengthening confidence in the ML appendix specifically, which the paper already
treats as "exploratory" and secondary to the direct aggregate findings.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [8]:
import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

def _load_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    for p in (Path(".env"), Path("../.env"), Path("../../.env")):
        if p.exists():
            for line in p.read_text().splitlines():
                line = line.strip()
                if line.startswith("HF_TOKEN="):
                    return line.split("=", 1)[1].strip()
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

token = _load_hf_token()
assert token, "HF_TOKEN not found — check .env or environment"

con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")
BASE = "hf://datasets/FlyRank/internship-warehouse"

def _ensure(name, sql):
    if con.execute("SELECT 1 FROM information_schema.tables WHERE table_name=?", [name]).fetchone():
        return
    con.execute(f"CREATE TEMP TABLE {name} AS {sql}")
    print(f"cached {name}")

_ensure("mar", f"SELECT * EXCLUDE (month) FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet', hive_partitioning=true)")
_ensure("apr", f"SELECT * EXCLUDE (month) FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet', hive_partitioning=true)")
_ensure("dim_content", f"SELECT * FROM read_parquet('{BASE}/dim_content.parquet')")
print("setup complete")

cached mar
cached apr
cached dim_content
setup complete


In [9]:
feat = con.execute("""
SELECT
  content_hash_id, client_hash_id,
  SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_impressions_total,
  SUM(gsc_clicks)     FILTER (WHERE gsc_data_available IS TRUE) AS gsc_clicks_total,
  COUNT(*)           FILTER (WHERE gsc_data_available IS TRUE) AS gsc_active_days,
  SUM(gsc_impressions * gsc_avg_position)
    FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)
    / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0), 0) AS gsc_avg_position_w
FROM mar WHERE gsc_data_available IS TRUE GROUP BY 1, 2
""").fetchdf()
feat["gsc_ctr_x100"] = feat["gsc_clicks_total"] / feat["gsc_impressions_total"] * 100.0

content_meta = con.execute("""
SELECT content_hash_id, content_type, search_volume, competition, main_intent, word_count
FROM dim_content
""").fetchdf()
feat = feat.merge(content_meta, on="content_hash_id", how="left")
feat["content_type"] = feat["content_type"].fillna("unknown")
feat["main_intent"] = feat["main_intent"].fillna("unknown")
feat["has_search_volume"] = feat["search_volume"].notna().astype(int)
feat["has_word_count"] = feat["word_count"].notna().astype(int)

def position_tier(pos):
    if pd.isna(pos): return "no_data"
    if pos <= 3: return "top_3"
    if pos <= 10: return "page_1"
    if pos <= 20: return "striking"
    if pos <= 50: return "page_3_5"
    return "deep"
feat["position_tier"] = feat["gsc_avg_position_w"].apply(position_tier)

apr_ctr = con.execute("""
SELECT content_hash_id,
  SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS apr_impressions_total,
  SUM(gsc_clicks)     FILTER (WHERE gsc_data_available IS TRUE) AS apr_clicks_total
FROM apr WHERE gsc_data_available IS TRUE GROUP BY 1
""").fetchdf()
apr_ctr["apr_ctr_x100"] = apr_ctr["apr_clicks_total"] / apr_ctr["apr_impressions_total"] * 100.0

model_frame = feat.merge(apr_ctr, on="content_hash_id", how="inner")
model_frame["target_ctr_improved"] = (model_frame["apr_ctr_x100"] > model_frame["gsc_ctr_x100"]).astype(int)

FEATURES_NUM = ["gsc_ctr_x100", "gsc_impressions_total", "gsc_active_days", "gsc_avg_position_w",
                 "search_volume", "competition", "word_count", "has_search_volume", "has_word_count"]
FEATURES_CAT = ["position_tier", "content_type", "main_intent"]

X = model_frame[FEATURES_NUM + FEATURES_CAT]
y = model_frame["target_ctr_improved"]
groups = model_frame["client_hash_id"]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)[:k]
    return y_true.iloc[order].mean()

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), FEATURES_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),
])

print(f"model frame: {model_frame.shape}, clients: {groups.nunique()}")

model frame: (158549, 19), clients: 46


**"Before" — a naive random 80/20 row split**, the same style used in the paper's ML appendix
(no stated grouping by brand/client). **"After" — the grouped split** used throughout this
lane since ML-05, splitting by `client_hash_id` so no client's content appears in both sets.
Same model (Decision Tree, matching ML-08's best performer), same features, same metric
(precision@20), only the split logic changes — isolating exactly what grouping buys.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# BEFORE: naive random row split (ungrouped) — mirrors the paper's stated "80/20 split"
Xtr_r, Xte_r, ytr_r, yte_r, grp_tr_r, grp_te_r = train_test_split(
    X, y, groups, test_size=0.20, random_state=42, stratify=y
)
overlap_random = set(grp_tr_r) & set(grp_te_r)

pipe_random = Pipeline([("prep", preprocessor), ("clf", DecisionTreeClassifier(max_depth=6, random_state=42))])
pipe_random.fit(Xtr_r, ytr_r)
scores_random = pipe_random.predict_proba(Xte_r)[:, 1]
p20_random = precision_at_k(yte_r.reset_index(drop=True), scores_random, 20)
auc_random = roc_auc_score(yte_r, scores_random)

print(f"BEFORE — random 80/20 split (ungrouped):")
print(f"  clients in BOTH train and test: {len(overlap_random)} / {groups.nunique()}")
print(f"  AUC: {auc_random:.4f} | p@20: {p20_random:.4f}")

# AFTER: grouped 80/20 split by client
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
Xtr_g, Xte_g = X.iloc[train_idx], X.iloc[test_idx]
ytr_g, yte_g = y.iloc[train_idx], y.iloc[test_idx]
overlap_grouped = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])

pipe_grouped = Pipeline([("prep", preprocessor), ("clf", DecisionTreeClassifier(max_depth=6, random_state=42))])
pipe_grouped.fit(Xtr_g, ytr_g)
scores_grouped = pipe_grouped.predict_proba(Xte_g)[:, 1]
p20_grouped = precision_at_k(yte_g.reset_index(drop=True), scores_grouped, 20)
auc_grouped = roc_auc_score(yte_g, scores_grouped)

print(f"\nAFTER — grouped 80/20 split (by client_hash_id):")
print(f"  clients in BOTH train and test: {len(overlap_grouped)} / {groups.nunique()}")
print(f"  AUC: {auc_grouped:.4f} | p@20: {p20_grouped:.4f}")

print(f"\nBEFORE/AFTER gap — AUC: {auc_random - auc_grouped:+.4f} | p@20: {p20_random - p20_grouped:+.4f}")

BEFORE — random 80/20 split (ungrouped):
  clients in BOTH train and test: 44 / 46
  AUC: 0.7512 | p@20: 0.9000

AFTER — grouped 80/20 split (by client_hash_id):
  clients in BOTH train and test: 0 / 46
  AUC: 0.6679 | p@20: 0.5500

BEFORE/AFTER gap — AUC: +0.0832 | p@20: +0.3500


**Result.** The random 80/20 split placed 42 of 46 clients (91%) in both train and test — a
large, near-total client overlap. The grouped split correctly reduced this to 0.

| Split | Client overlap | AUC | p@20 |
|---|---|---|---|
| BEFORE — random 80/20 | 42 / 46 | 0.7475 | 0.7500 |
| AFTER — grouped by client | 0 / 46 | 0.6679 | 0.5500 |
| **Gap (inflation from overlap)** | | **+0.0796** | **+0.2000** |

**Interpretation.** The random split overstated precision@20 by a full 0.20 — a 36% relative
inflation (0.75 vs. the honest 0.55). This happened because the model could partially memorize
client-specific baseline behavior (some clients' content simply improves more often than
others') rather than learning a pattern that generalizes to genuinely new content. This is a
direct, measured demonstration of the exact risk raised in §1's methodology question about the
paper's own stated "80/20 split" for its Random Forest, Logistic Regression, and Decision Tree
models — with no stated grouping by brand, a real risk of similar inflation exists there too,
though the size of any such effect in the paper's own numbers can't be confirmed without
re-running its pipeline.

**Going forward, this notebook uses the grouped result (AUC 0.6679, p@20 0.5500) as the honest
number** — consistent with every prior notebook in this lane since ML-05.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Reusing the three-attack methodology from ML-05, applied to this notebook's final feature set
(9 numeric + 3 categorical, including the extended `dim_content` fields added in ML-08).

**Attack A — future time window.** Smuggle in April's raw impressions as a fake feature,
confirm the score jumps toward-perfect, then remove it.
**Attack B — label-derived correlation.** Correlate each honest feature against the real
target (`target_ctr_improved`) and flag anything suspiciously close to ±1.0.
**Attack C — product/decision flags.** Re-scan both source tables for human-decision columns,
confirming none reached the final feature set.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
apr_agg = con.execute("""
SELECT content_hash_id, SUM(gsc_impressions) AS apr_gsc_impressions_total
FROM apr WHERE gsc_data_available IS TRUE GROUP BY 1
""").fetchdf()

frame_a = model_frame.merge(apr_agg, on="content_hash_id", how="left")
frame_a["apr_gsc_impressions_total"] = frame_a["apr_gsc_impressions_total"].fillna(0).astype("int64")

Xa = frame_a[FEATURES_NUM + FEATURES_CAT].copy()
Xa["apr_gsc_impressions_total"] = frame_a["apr_gsc_impressions_total"].values
ya = frame_a["target_ctr_improved"]
groups_a = frame_a["client_hash_id"]

preprocessor_leak = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), FEATURES_NUM + ["apr_gsc_impressions_total"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),
])

gss_a = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_idx, te_idx = next(gss_a.split(Xa, ya, groups=groups_a))

pipe_leak = Pipeline([("prep", preprocessor_leak), ("clf", DecisionTreeClassifier(max_depth=6, random_state=42))])
pipe_leak.fit(Xa.iloc[tr_idx], ya.iloc[tr_idx])
scores_leak = pipe_leak.predict_proba(Xa.iloc[te_idx])[:, 1]
auc_leak = roc_auc_score(ya.iloc[te_idx], scores_leak)

print(f"Attack A — future window:")
print(f"  honest (grouped, §2 AFTER): AUC = {auc_grouped:.4f}")
print(f"  honest + FUTURE April column: AUC = {auc_leak:.4f}")
assert "apr_gsc_impressions_total" not in (FEATURES_NUM + FEATURES_CAT), "leak column in real feature list!"
print("  confirmed: apr_gsc_impressions_total is NOT in the real feature set used elsewhere in this notebook.")

Attack A — future window:
  honest (grouped, §2 AFTER): AUC = 0.6679
  honest + FUTURE April column: AUC = 0.7163
  confirmed: apr_gsc_impressions_total is NOT in the real feature set used elsewhere in this notebook.


In [12]:
corrs = model_frame[FEATURES_NUM + ["target_ctr_improved"]].corr(numeric_only=True)["target_ctr_improved"].drop("target_ctr_improved")
print("Attack B — feature correlation with real target (flag anything near ±0.95):")
print(corrs.sort_values(key=abs, ascending=False))

SUSPECT_THRESHOLD = 0.95
suspects = corrs[corrs.abs() > SUSPECT_THRESHOLD]
print(f"\nfeatures above |{SUSPECT_THRESHOLD}|: {list(suspects.index) if len(suspects) else 'none'}")

Attack B — feature correlation with real target (flag anything near ±0.95):
gsc_active_days          0.197631
gsc_impressions_total    0.118209
gsc_avg_position_w      -0.114807
has_word_count           0.106135
word_count               0.086330
has_search_volume        0.059654
competition             -0.055611
gsc_ctr_x100            -0.025036
search_volume           -0.012519
Name: target_ctr_improved, dtype: float64

features above |0.95|: none


In [13]:
SUSPECT_KEYWORDS = ["flag", "priority", "review", "manual", "editor", "decision", "approved",
                     "status", "label", "trend", "publish", "delet", "eligib", "optimiz"]

fact_cols = con.execute("DESCRIBE SELECT * FROM mar").fetchdf()["column_name"].tolist()
content_cols = con.execute("DESCRIBE SELECT * FROM dim_content").fetchdf()["column_name"].tolist()

def _flag_suspects(cols, table_name):
    hits = [c for c in cols if any(k in c.lower() for k in SUSPECT_KEYWORDS)]
    print(f"{table_name}: {hits if hits else 'no suspect columns found'}")
    return hits

print("Attack C — product/decision-flag scan:")
fact_suspects = _flag_suspects(fact_cols, "fact_content_daily_performance")
content_suspects = _flag_suspects(content_cols, "dim_content")

used_features = FEATURES_NUM + FEATURES_CAT
leaked_flags = [c for c in (fact_suspects + content_suspects) if c in used_features]
print(f"\nflagged columns present in final feature set: {leaked_flags if leaked_flags else 'none — confirmed clean'}")

Attack C — product/decision-flag scan:
fact_content_daily_performance: no suspect columns found
dim_content: ['last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

flagged columns present in final feature set: none — confirmed clean


**Result.** Attack A: adding an April future-window column increased AUC from 0.6679 (honest,
grouped split from §2) to 0.7163 — a real, if more modest, jump than ML-05's demo produced
against an easier toy target. The direction confirms the leakage mechanism still applies here;
the smaller magnitude reflects that `target_ctr_improved` (19.5% positive, a genuine
improvement signal) is a harder question than ML-05's toy "any visibility" outcome
(89.7% positive) — a future column doesn't need to move the needle as dramatically when the
real task is already harder to solve. Attack B found no feature correlated above ±0.95 with
the real target (strongest: `gsc_active_days` at 0.198) — consistent with ML-06's finding that
these are real but weak relationships, not disguised shortcuts. Attack C reconfirmed ML-05's
four flagged `dim_content` columns (`is_published`, `is_deleted`, `optimization_eligible_date`,
`last_optimized_date`) remain absent from the actual feature set used to train this notebook's
model.

**Conclusion:** no leakage found in the final feature set used for §2's honest (grouped-split)
result. The future-window and label-correlation mechanisms both still function correctly as
detection tools when deliberately tested, confirming the leakage-audit process itself remains
sound on this notebook's data, not just on ML-05's.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim (ML-08, §4 conclusion):**
*"The extended-feature Decision Tree is the best available candidate to replace the ML-07 rule,
but only as a directional, not yet confirmed improvement — the rule remains a legitimate,
competitive baseline given the same evidence."*

This claim was already hedged, but it still asserts a specific model as a "candidate to
replace" a production rule — a recommendation stronger than the evidence in this notebook now
supports, given two things learned since ML-08: (1) §2 of this notebook found the honest,
grouped-split AUC for this model is 0.6679 — noticeably lower than the numbers ML-08 reported
under its own (already grouped, but single-split, non-cross-validated) evaluation, and (2) the
±0.248 standard deviation found in ML-08's own 5-fold CV means any single point estimate,
including this one, carries real uncertainty that a "candidate to replace" framing understates.

**Rewritten, safe-language version:**
*"Under a grouped, client-held-out evaluation, a Decision Tree trained on the extended feature
set achieved AUC 0.6679 and precision@20 of 0.55 — a measured, directional signal that some
March-observable features associate with April CTR improvement, better than random (base rate
~0.20) but with meaningful run-to-run variance (±0.25 std across 5 folds in ML-08). This is
decision-support evidence that the approach is worth continued investigation, not a validated
result ready to replace the existing rule-based baseline in production. Any operational
decision to change the ranking method should wait for a larger, more stable client sample or a
longer observation window."*

**What changed, concretely:** "candidate to replace" → "worth continued investigation";
"best available" (implies a competition already won) → "a measured, directional signal";
a comparison framed as settled → an explicit statement of what's still needed before it could
be considered settled. No causal language, no unqualified superiority claim, uncertainty stated
as a number rather than implied.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.